# Experiment 8a — TiDE (Time-series Dense Encoder), proper-tier training

Pure-MLP seq2seq from Google Research (Das et al. 2023, *Long-term Forecasting with TiDE*). Phase 8's question: *can we close the L2-metric gap to Chronos and challenge for Overall rank 1*, after Phase 7 produced two submissions that lead on WAPE but trail on MSE/RMSE/MAPE?

**This notebook replaces an earlier diagnostic-tier run** (96 windows/series/epoch, 30 epochs, dropout 0.2, 2 decoder blocks) that landed at CV-WAPE 16.4 and was almost certainly under-trained — all 9 (fold × seed) runs converged to inner-val L1 in the band 0.305-0.326, classic shallow-attractor / under-fitting signature for a 7.88M-parameter model fed only 276K total training samples. This version brings the training scale into line with the model capacity:

| Hyperparameter | Diagnostic v1 | Proper v2 (this) |
|---|---|---|
| `WINDOWS_PER_SERIES_PER_EPOCH` | 96 | **384** |
| `EPOCHS` | 30 | **100** |
| `EARLY_STOP` patience | 5 | **15** |
| `DEC_PER_STEP` | 8 | **16** |
| `N_DEC_BLOCKS` | 2 | **3** |
| `DROPOUT` | 0.2 | **0.3** |
| `BATCH` | 64 | **128** |
| LR schedule | Cosine T_max=30 | **5-epoch linear warmup → cosine T_max=100** |
| Total training samples seen | ~276K | **~3.7M** (13×) |

Expected runtime: **~15-20 min per fold/seed**, **~2-2.5 hours total** on T4. Progress prints every 10 epochs so convergence is visible.

Decision shape unchanged from v1: beat 7e-gated WAPE (14.59) AND beat OOF MSE 8.30 to be a real rank-1 candidate; partial beats are partial results; r < 0.85 makes TiDE a blend candidate even if solo is worse; complete failure means scaling alone wasn't enough.

---
## §0. Configuration

In [ ]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-8")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336
L_PAST       = 168

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]

BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]

# ── TiDE config — proper-tier ────────────────────────────────────────────────
FEAT_PROJ_DIM   = 8
HIDDEN          = 128
DEC_PER_STEP    = 16        # ↑ from 8 — wider per-step state
EMBED_DIM       = 16
N_ENC_BLOCKS    = 2
N_DEC_BLOCKS    = 3         # ↑ from 2
DROPOUT         = 0.3       # ↑ from 0.2 — more reg with more training

# Training
BATCH                        = 128       # ↑ from 64
LR                           = 1e-3
WEIGHT_DECAY                 = 1e-4
EPOCHS                       = 100       # ↑ from 30
EARLY_STOP                   = 15        # ↑ from 5
WARMUP_EPOCHS                = 5
WINDOWS_PER_SERIES_PER_EPOCH = 384       # ↑ from 96 — 4× more training data per epoch
PROGRESS_EVERY               = 10        # print val loss every N epochs
SEEDS                        = [42, 101, 202]

# Decision thresholds — pre-registered
BEATS_MLP_THRESHOLD   = 14.52
BEATS_GATED_THRESHOLD = 14.59
BEATS_LGBM_THRESHOLD  = 14.78
MSE_TARGET_OOF        = 8.30

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"TiDE arch   : L_past={L_PAST}, H={EVAL_HORIZON}, feat_proj={FEAT_PROJ_DIM}, hidden={HIDDEN},")
print(f"              dec_per_step={DEC_PER_STEP}, enc/dec blocks={N_ENC_BLOCKS}/{N_DEC_BLOCKS}, dropout={DROPOUT}")
print(f"Train       : batch {BATCH}, lr {LR}, epochs {EPOCHS} (early stop {EARLY_STOP}, warmup {WARMUP_EPOCHS}),")
print(f"              {WINDOWS_PER_SERIES_PER_EPOCH} windows/series/epoch, seeds {SEEDS}")
samples_per_epoch = WINDOWS_PER_SERIES_PER_EPOCH * 96
print(f"              → {samples_per_epoch:,} train samples/epoch, up to {samples_per_epoch * EPOCHS:,} total per seed")
print(f"Decisions   : beat gated-MLP @ {BEATS_GATED_THRESHOLD}, MSE target @ OOF {MSE_TARGET_OOF}")

---
## §1. Setup & data loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import math

import numpy as np
import pandas as pd
import lightgbm as lgb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["timestamp"])
sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
code_to_sid = {v: k for k, v in sid_to_code.items()}
N_SERIES = len(sid_to_code)

print(f"train_df: {train_df.shape}, # series: {N_SERIES}")

---
## §2. Sequence prep + tabular pipeline for the LGBM anchor

In [ ]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

base_tab = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_tab["sid_code"] = base_tab["series_id"].map(sid_to_code).astype("int32")
base_tab, TEMPORAL_FEATS = add_temporal_features(
    base_tab, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)
NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]

base_seq = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_seq["sid_code"] = base_seq["series_id"].map(sid_to_code).astype("int32")
base_seq = base_seq.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

SEQ_FEATURE_COLS = DOMAIN_FEAT_COLS + TIME_COLS + ["target"]
N_COV  = len(DOMAIN_FEAT_COLS)
N_TIME = len(TIME_COLS)
PAST_FEAT_PER_STEP = N_COV + N_TIME + 1
FUT_FEAT_PER_STEP  = N_COV + N_TIME

series_tensors_full = {}
for sid, code in sid_to_code.items():
    g = base_seq[base_seq["sid_code"] == code].sort_values("timestamp")
    series_tensors_full[code] = g[SEQ_FEATURE_COLS].to_numpy(dtype=np.float32)
T_PER_SERIES = next(iter(series_tensors_full.values())).shape[0]
print(f"Per-series tensor: ({T_PER_SERIES}, {PAST_FEAT_PER_STEP})")

---
## §3. CV framework + LGBM-MAE anchor

In [ ]:
def fold_split_tabular(end_off, length=EVAL_HORIZON):
    tr_list, te_list = [], []
    for _, g in base_tab.groupby("series_id", sort=False):
        g = g.sort_values("timestamp")
        cut = len(g) - end_off
        tr_list.append(g.iloc[:cut])
        te_list.append(g.iloc[cut:cut + length])
    tr = pd.concat(tr_list, ignore_index=True)
    te = pd.concat(te_list, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

def fold_offset(k):
    return (N_FOLDS - k) * EVAL_HORIZON

def fold_train_end_idx(k):
    return T_PER_SERIES - fold_offset(k)

def fit_lgbm_predict(tr, te, feats, params, objective, categorical=None, seed=SEED):
    model = lgb.LGBMRegressor(**params, objective=objective, random_state=seed)
    fit_kw = {} if not categorical else {"categorical_feature": categorical}
    model.fit(tr[feats], tr["target"].to_numpy(float), **fit_kw)
    return model.predict(te[feats]), model

print("Running LightGBM-MAE anchor...")
oof_lgbm_rows, per_fold_lgbm = [], []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    preds, _ = fit_lgbm_predict(tr, te, FEATS_WITH_SID, BASELINE_LGBM,
                                "regression_l1", categorical=["sid_code"])
    per_fold_lgbm.append(wape(te["target"], preds))
    oof_lgbm_rows.append(te.assign(pred_lgbm=preds, fold=k)[
        ["series_id", "timestamp", "target", "pred_lgbm", "fold"]
    ])
oof_lgbm = pd.concat(oof_lgbm_rows, ignore_index=True)
lgbm_arr = np.array(per_fold_lgbm)
print(f"  mean LGBM CV-WAPE = {lgbm_arr.mean():.3f}")
assert abs(lgbm_arr.mean() - 14.78) < 0.5
print("✅ LGBM anchor reproduces.")

---
## §4. TiDE architecture

Architecture code unchanged from v1 — only the §0 constants `DEC_PER_STEP` (8 → 16) and `N_DEC_BLOCKS` (2 → 3) change, which the same `TiDE` class handles.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout, hidden_ratio=2):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1  = nn.Linear(dim, dim * hidden_ratio)
        self.fc2  = nn.Linear(dim * hidden_ratio, dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm(x)
        h = self.fc1(h); h = F.relu(h); h = self.drop(h)
        h = self.fc2(h); h = self.drop(h)
        return x + h


class TiDE(nn.Module):
    def __init__(self, l_past, h_future, n_past_feat, n_future_feat,
                 n_series, embed_dim, feat_proj_dim, hidden, dec_per_step,
                 n_enc_blocks, n_dec_blocks, dropout):
        super().__init__()
        self.l_past, self.h_future = l_past, h_future
        self.dec_per_step = dec_per_step
        self.past_proj   = nn.Linear(n_past_feat, feat_proj_dim)
        self.future_proj = nn.Linear(n_future_feat, feat_proj_dim)
        self.series_emb  = nn.Embedding(n_series, embed_dim)
        self.enc_in      = nn.Linear(l_past * feat_proj_dim + embed_dim, hidden)
        self.encoder     = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(n_enc_blocks)])
        self.dec_in      = nn.Linear(hidden + h_future * feat_proj_dim, h_future * dec_per_step)
        self.decoder     = nn.Sequential(*[ResidualBlock(dec_per_step, dropout) for _ in range(n_dec_blocks)])
        self.temporal_dec = nn.Linear(dec_per_step + feat_proj_dim, 1)

    def forward(self, past_seq, future_seq, sid):
        B = past_seq.size(0)
        past_p = self.past_proj(past_seq)
        fut_p  = self.future_proj(future_seq)
        past_flat = past_p.reshape(B, -1)
        sid_e = self.series_emb(sid)
        enc = self.enc_in(torch.cat([past_flat, sid_e], dim=-1))
        enc = self.encoder(enc)
        fut_flat = fut_p.reshape(B, -1)
        dec = self.dec_in(torch.cat([enc, fut_flat], dim=-1))
        dec = dec.reshape(B, self.h_future, self.dec_per_step)
        dec = self.decoder(dec)
        td_in = torch.cat([dec, fut_p], dim=-1)
        return self.temporal_dec(td_in).squeeze(-1)

    def feature_importance(self):
        with torch.no_grad():
            W = self.past_proj.weight.detach().cpu()
            return torch.norm(W, dim=0).numpy()


_m = TiDE(L_PAST, EVAL_HORIZON, PAST_FEAT_PER_STEP, FUT_FEAT_PER_STEP,
          N_SERIES, EMBED_DIM, FEAT_PROJ_DIM, HIDDEN, DEC_PER_STEP,
          N_ENC_BLOCKS, N_DEC_BLOCKS, DROPOUT)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"TiDE parameters: {n_params:,}")
del _m

---
## §5. Training utilities — with warmup-cosine schedule and per-epoch progress

Two notable changes from v1:

- **`make_warmup_cosine`** — 5-epoch linear warmup from 10%→100% of base LR, then cosine decay over the remaining 95 epochs. Standard transformer-era schedule. v1's cosine-only schedule with `T_max=30` was decaying LR to ~0 before the 7.88M-param model had a chance to refine.
- **`train_tide_one_seed`** prints `epoch | train_L1 | val_L1 | lr` every `PROGRESS_EVERY` epochs and on every new-best val loss, so convergence is visible in the notebook rather than waiting blind for 15 minutes per seed.

In [ ]:
from sklearn.preprocessing import RobustScaler

def fit_per_series_target_stats(tensor_by_sid, train_end_idx):
    mus, sds = {}, {}
    for sid, arr in tensor_by_sid.items():
        tgt = arr[:train_end_idx, -1]
        mus[sid] = float(np.mean(tgt))
        sds[sid] = float(max(np.std(tgt), 1e-3))
    return mus, sds

def fit_feature_scaler(tensor_by_sid, train_end_idx):
    rows = np.concatenate([arr[:train_end_idx] for arr in tensor_by_sid.values()], axis=0)
    return RobustScaler().fit(np.nan_to_num(rows, nan=0.0))

def apply_scalers(arr, feat_scaler, sid_mu, sid_sd):
    out = feat_scaler.transform(np.nan_to_num(arr, nan=0.0)).astype(np.float32)
    raw_tgt = np.nan_to_num(arr[:, -1], nan=0.0)
    out[:, -1] = ((raw_tgt - sid_mu) / sid_sd).astype(np.float32)
    return out

def valid_anchors(train_end_idx):
    return L_PAST, train_end_idx - EVAL_HORIZON

def sample_train_anchors(train_end_idx, rng, windows_per_series, val_split=0.1):
    lo, hi = valid_anchors(train_end_idx)
    if hi <= lo: return [], []
    cut = int(lo + (hi - lo) * (1 - val_split))
    train_anchors, val_anchors = [], []
    for sid in range(N_SERIES):
        t_train = rng.integers(lo, cut, size=windows_per_series)
        train_anchors.extend([(sid, int(t)) for t in t_train])
        n_val = max(1, windows_per_series // 4)
        if hi + 1 > cut:
            t_val = rng.integers(cut, hi + 1, size=n_val)
            val_anchors.extend([(sid, int(t)) for t in t_val])
    return train_anchors, val_anchors


class TiDEDataset(Dataset):
    def __init__(self, anchors, scaled_tensors):
        self.anchors = anchors
        self.tens = scaled_tensors
    def __len__(self): return len(self.anchors)
    def __getitem__(self, i):
        sid, t = self.anchors[i]
        arr = self.tens[sid]
        past = arr[t - L_PAST : t]
        future_cov = arr[t : t + EVAL_HORIZON, :N_COV + N_TIME]
        future_target = arr[t : t + EVAL_HORIZON, -1]
        return (torch.from_numpy(past),
                torch.from_numpy(future_cov),
                torch.tensor(sid, dtype=torch.long),
                torch.from_numpy(future_target))


def make_warmup_cosine(opt, warmup_epochs, total_epochs, start_factor=0.1):
    warmup = LinearLR(opt, start_factor=start_factor, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(opt, T_max=max(1, total_epochs - warmup_epochs))
    return SequentialLR(opt, schedulers=[warmup, cosine], milestones=[warmup_epochs])


def train_tide_one_seed(scaled_tensors, sid_mu, sid_sd, train_end_idx, seed, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    model = TiDE(L_PAST, EVAL_HORIZON, PAST_FEAT_PER_STEP, FUT_FEAT_PER_STEP,
                 N_SERIES, EMBED_DIM, FEAT_PROJ_DIM, HIDDEN, DEC_PER_STEP,
                 N_ENC_BLOCKS, N_DEC_BLOCKS, DROPOUT).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_warmup_cosine(opt, WARMUP_EPOCHS, EPOCHS)
    loss_fn = nn.L1Loss()

    best_val, patience, best_state = float("inf"), 0, None
    history = []

    for epoch in range(EPOCHS):
        train_anchors, val_anchors = sample_train_anchors(train_end_idx, rng, WINDOWS_PER_SERIES_PER_EPOCH)
        if not train_anchors:
            return None, None, float("inf"), []
        tr_loader = DataLoader(TiDEDataset(train_anchors, scaled_tensors),
                               batch_size=BATCH, shuffle=True, num_workers=0)
        val_loader = DataLoader(TiDEDataset(val_anchors, scaled_tensors),
                                batch_size=BATCH, shuffle=False, num_workers=0)

        model.train()
        train_losses = []
        for past, fut, sid, y in tr_loader:
            past, fut, sid, y = past.to(DEVICE), fut.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred = model(past, fut, sid)
            loss = loss_fn(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_losses.append(loss.item())
        sched.step()
        train_l1 = float(np.mean(train_losses))

        model.eval()
        with torch.no_grad():
            vlosses = []
            for past, fut, sid, y in val_loader:
                past, fut, sid, y = past.to(DEVICE), fut.to(DEVICE), sid.to(DEVICE), y.to(DEVICE)
                vlosses.append(loss_fn(model(past, fut, sid), y).item())
            val_loss = float(np.mean(vlosses)) if vlosses else float("inf")

        history.append((epoch + 1, train_l1, val_loss))
        cur_lr = opt.param_groups[0]["lr"]

        improved = val_loss < best_val - 1e-5
        if improved:
            best_val, patience = val_loss, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience += 1

        if verbose and ((epoch + 1) % PROGRESS_EVERY == 0 or epoch == 0 or improved):
            marker = " ⭢ new best" if improved else ""
            print(f"      epoch {epoch+1:>3d}: train_L1={train_l1:.4f}  val_L1={val_loss:.4f}  lr={cur_lr:.2e}{marker}")

        if patience >= EARLY_STOP:
            if verbose: print(f"      early stop at epoch {epoch+1} (patience {EARLY_STOP})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    oof = np.zeros((N_SERIES, EVAL_HORIZON), dtype=np.float32)
    with torch.no_grad():
        for sid in range(N_SERIES):
            arr = scaled_tensors[sid]
            past = torch.tensor(arr[train_end_idx - L_PAST : train_end_idx][None, ...],
                                dtype=torch.float32, device=DEVICE)
            fut_cov = torch.tensor(
                arr[train_end_idx : train_end_idx + EVAL_HORIZON, :N_COV + N_TIME][None, ...],
                dtype=torch.float32, device=DEVICE)
            sid_t = torch.tensor([sid], dtype=torch.long, device=DEVICE)
            pred_norm = model(past, fut_cov, sid_t).cpu().numpy()[0]
            oof[sid] = pred_norm * sid_sd[sid] + sid_mu[sid]
    oof = np.clip(oof, 0.0, None)
    return oof, model.feature_importance(), best_val, history

print("TiDE proper-tier training utilities ready.")

---
## §6. CV run — 3 folds × 3 seeds, proper-tier training

Expect each (fold, seed) to take ~15-20 min on T4. Per-epoch progress lines (every 10 epochs, plus on every new-best val loss) let you see whether training is genuinely converging or stuck.

In [ ]:
import time

oof_tide_rows, per_fold_wape, per_fold_seed_spread = [], [], []
fi_per_run = []
history_per_run = []

for k in range(N_FOLDS):
    fold_t0 = time.time()
    end_idx = fold_train_end_idx(k)
    print(f"\n══════ Fold {k}: train_end_idx = {end_idx}, test = [{end_idx}, {end_idx + EVAL_HORIZON}) ══════")

    sid_mu, sid_sd = fit_per_series_target_stats(series_tensors_full, end_idx)
    feat_scaler = fit_feature_scaler(series_tensors_full, end_idx)
    scaled = {sid: apply_scalers(arr, feat_scaler, sid_mu[sid], sid_sd[sid])
              for sid, arr in series_tensors_full.items()}

    seed_oofs, seed_wapes = [], []
    for s in SEEDS:
        seed_t0 = time.time()
        print(f"\n   ── seed {s} ──")
        oof_pred, fi, val_loss, history = train_tide_one_seed(
            scaled, sid_mu, sid_sd, end_idx, s, verbose=True
        )
        seed_oofs.append(oof_pred)
        fi_per_run.append((k, s, fi))
        history_per_run.append((k, s, history))

        test_true = np.stack([series_tensors_full[sid][end_idx : end_idx + EVAL_HORIZON, -1]
                              for sid in range(N_SERIES)], axis=0)
        sw = wape(test_true.ravel(), oof_pred.ravel())
        seed_wapes.append(sw)
        print(f"   ── seed {s}: WAPE = {sw:.3f}, best_val_L1 = {val_loss:.4f}, "
              f"epochs run = {len(history)}, time = {time.time() - seed_t0:.1f}s")

    avg_oof = np.mean(np.stack(seed_oofs, axis=0), axis=0)
    test_true = np.stack([series_tensors_full[sid][end_idx : end_idx + EVAL_HORIZON, -1]
                          for sid in range(N_SERIES)], axis=0)
    fold_w = wape(test_true.ravel(), avg_oof.ravel())
    spread = max(seed_wapes) - min(seed_wapes)
    per_fold_wape.append(fold_w)
    per_fold_seed_spread.append(spread)
    print(f"\n   ══ fold {k}: avg WAPE = {fold_w:.3f}, seed spread = {spread:.3f}, "
          f"time = {time.time() - fold_t0:.1f}s ══")

    test_ts = base_seq["timestamp"].to_numpy().reshape(N_SERIES, T_PER_SERIES)[:, end_idx : end_idx + EVAL_HORIZON]
    for sid in range(N_SERIES):
        oof_tide_rows.append(pd.DataFrame({
            "series_id": code_to_sid[sid],
            "timestamp": pd.to_datetime(test_ts[sid]),
            "target":    test_true[sid],
            "pred_tide": avg_oof[sid],
            "lead_h":    np.arange(1, EVAL_HORIZON + 1),
            "fold":      k,
        }))

oof_tide = pd.concat(oof_tide_rows, ignore_index=True)
tide_arr = np.array(per_fold_wape)

print("\n\n══════ TiDE CV summary ══════")
print(f"{'fold':>6}  {'WAPE':>8}  {'seed spread':>12}")
for k, (w, sp) in enumerate(zip(tide_arr, per_fold_seed_spread)):
    print(f"  {k:>4}    {w:>7.3f}    {sp:>10.3f}")
print(f"  mean    {tide_arr.mean():>7.3f}")
print(f"  std     {tide_arr.std():>7.3f}")

print(f"\nAnchors  — LGBM: {lgbm_arr.mean():.3f}  |  7a-MLP: {BEATS_MLP_THRESHOLD:.2f}  |  7e-gated: {BEATS_GATED_THRESHOLD:.2f}")
print(f"TiDE solo (§6)     : {tide_arr.mean():.3f}")
print(f"Δ vs LGBM          : {tide_arr.mean() - lgbm_arr.mean():+.3f}")
print(f"Δ vs 7a-MLP        : {tide_arr.mean() - BEATS_MLP_THRESHOLD:+.3f}")
print(f"Δ vs 7e-gated      : {tide_arr.mean() - BEATS_GATED_THRESHOLD:+.3f}")

---
## §7. Headline — full metric panel + residual r + feature importance

The MSE column is what matters for the Chronos question. Feature importance is from `past_proj` column norms; v1 had `target_history` tied with `queue_pressure_forecast` at the top, which we attributed to autoregressive leaning — with proper training, worth re-checking whether reliance on past target moderates.

In [ ]:
def all_metrics(y, p):
    err = y - p
    return dict(
        MAE=float(np.mean(np.abs(err))),
        MSE=float(np.mean(err ** 2)),
        RMSE=float(np.sqrt(np.mean(err ** 2))),
        WAPE=float(100 * np.sum(np.abs(err)) / np.sum(np.abs(y))),
    )

joined = oof_lgbm.merge(
    oof_tide[["series_id", "timestamp", "pred_tide", "lead_h", "fold"]],
    on=["series_id", "timestamp", "fold"], how="inner",
)
assert len(joined) == len(oof_lgbm) == len(oof_tide), "OOF row counts diverged."

y = joined["target"].to_numpy()
rows = [
    {"model": "LGBM-MAE", **all_metrics(y, joined["pred_lgbm"].to_numpy())},
    {"model": "TiDE v2",  **all_metrics(y, joined["pred_tide"].to_numpy())},
]
metric_df = pd.DataFrame(rows)
print("=== OOF metrics ===\n")
print(metric_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Reference ---")
print("  OOF: 7a-MLP MSE 8.77 WAPE 14.47  |  7e-gated MSE 8.75 WAPE 14.54")
print("  LB:  chronos MSE 7.72 WAPE 14.81  |  ensemble MSE 9.39 WAPE 14.68  |  gated-mlp MSE 9.37 WAPE 14.75")
print("  v1 TiDE (under-trained): MSE 9.48 WAPE 16.37")

joined["resid_lgbm"] = joined["target"] - joined["pred_lgbm"]
joined["resid_tide"] = joined["target"] - joined["pred_tide"]
r_pooled = float(np.corrcoef(joined["resid_lgbm"], joined["resid_tide"])[0, 1])
r_per_fold = [float(np.corrcoef(joined.loc[joined.fold == k, "resid_lgbm"],
                                joined.loc[joined.fold == k, "resid_tide"])[0, 1])
              for k in range(N_FOLDS)]
print("\n=== Pearson r(LGBM resid, TiDE resid) ===")
for k, r in enumerate(r_per_fold):
    print(f"  fold {k}: {r:.4f}")
print(f"  pooled : {r_pooled:.4f}")

ALL_PAST_FEAT_NAMES = DOMAIN_FEAT_COLS + TIME_COLS + ["target_history"]
fi_matrix = np.stack([f for (_, _, f) in fi_per_run], axis=0)
fi_mean = fi_matrix.mean(axis=0)
fi_std  = fi_matrix.std(axis=0)
fi_normed = fi_mean / fi_mean.max()
order = np.argsort(-fi_normed)
print("\n=== TiDE input projection norms (implicit feature importance, 9 fold×seed runs) ===")
print(f"  {'feature':<40}  {'norm':>6}  {'rel':>6}  {'std':>6}")
print("  " + "-" * 70)
for i in order:
    print(f"  {ALL_PAST_FEAT_NAMES[i]:<40}  {fi_mean[i]:>6.3f}  {fi_normed[i]:>6.3f}  {fi_std[i]:>6.3f}")

print("\n  Cross-check vs 7e gate ordering:")
print("  7e top    : queue_pressure > event_load > network_pressure")
print("  7e bottom : workload_intensity > service_irreg_risk > unit_reliability")
print("  v1 TiDE   : target_history tied with queue_pressure at top — autoregressive lean.")
print("  This run  : does target_history rank stay at top, or drop after proper training?")

---
## §8. Per-horizon WAPE

v1 had a striking +8.89 WAPE win at lead-1h and ~−1.5 to −2 WAPE losses at every other bucket — pure autoregressive-leaning behaviour. With proper training, the question is whether TiDE learns to use future covariates more aggressively and flattens the per-horizon curve.

In [ ]:
BINS = [(1, 1), (2, 6), (7, 24), (25, 72), (73, 168), (169, 336)]
print(f"{'lead':>10}  {'n':>6}  {'LGBM':>7}  {'TiDE':>7}  {'Δ_vs_LGBM':>10}")
print("-" * 50)
for lo, hi in BINS:
    sl = joined[(joined["lead_h"] >= lo) & (joined["lead_h"] <= hi)]
    if len(sl) == 0: continue
    wl = wape(sl["target"], sl["pred_lgbm"])
    wt = wape(sl["target"], sl["pred_tide"])
    print(f"  {lo:>3}..{hi:<3}h  {len(sl):>6}  {wl:>6.3f}  {wt:>6.3f}  {wl - wt:>+9.3f}")

print("\nReference (v1 under-trained):")
print("     1..1  h:  LGBM 24.05  TiDE 15.16  Δ +8.89")
print("     2..6  h:  LGBM 21.57  TiDE 21.44  Δ +0.14")
print("     7..24 h:  LGBM 12.51  TiDE 14.00  Δ -1.49")
print("    25..72 h:  LGBM 17.18  TiDE 19.25  Δ -2.07")
print("    73..168h:  LGBM 14.05  TiDE 15.98  Δ -1.92")
print("   169..336h:  LGBM 14.67  TiDE 16.13  Δ -1.46")

---
## §9. Save artifacts + decision summary

In [ ]:
oof_path = OUTPUT_DIR / "oof_predictions_8a_tide.csv"
oof_tide[["series_id", "timestamp", "target", "pred_tide", "lead_h", "fold"]].to_csv(oof_path, index=False)
print(f"OOF predictions → {oof_path}")

fi_rows = []
for (k, s, fi) in fi_per_run:
    for i, name in enumerate(ALL_PAST_FEAT_NAMES):
        fi_rows.append({"fold": k, "seed": s, "feature": name, "proj_norm": float(fi[i])})
fi_path = OUTPUT_DIR / "feature_importance_8a_tide.csv"
pd.DataFrame(fi_rows).to_csv(fi_path, index=False)
print(f"Feature importance → {fi_path}")

hist_rows = []
for (k, s, hist) in history_per_run:
    for (epoch, train_l1, val_l1) in hist:
        hist_rows.append({"fold": k, "seed": s, "epoch": epoch,
                          "train_L1": train_l1, "val_L1": val_l1})
hist_path = OUTPUT_DIR / "training_history_8a_tide.csv"
pd.DataFrame(hist_rows).to_csv(hist_path, index=False)
print(f"Training history → {hist_path}")

TIDE_MSE = float(all_metrics(y, joined["pred_tide"].to_numpy())["MSE"])
print("\n=== Decision summary ===")
print(f"  LGBM-MAE          : WAPE {lgbm_arr.mean():.3f}")
print(f"  TiDE v2 solo      : WAPE {tide_arr.mean():.3f}  ({tide_arr.mean() - BEATS_GATED_THRESHOLD:+.3f} vs 7e-gated)")
print(f"  TiDE v2 OOF MSE   : {TIDE_MSE:.4f}   (target for Chronos competitive: ≤ {MSE_TARGET_OOF})")
print(f"  Pearson r vs LGBM : {r_pooled:.4f}")

wape_beats_gated = tide_arr.mean() <= BEATS_GATED_THRESHOLD
mse_competitive  = TIDE_MSE <= MSE_TARGET_OOF
blend_useful     = r_pooled < 0.85

print("\n=== Verdict ===")
if wape_beats_gated and mse_competitive:
    print("✅ TiDE beats gated-MLP on WAPE AND has Chronos-competitive MSE.")
    print("   Write 8b submission notebook — refit on full train, 5 seeds, submit.")
elif wape_beats_gated:
    print("~  TiDE beats gated-MLP on WAPE but MSE is still off Chronos pace.")
    print(f"   ({TIDE_MSE:.3f} vs target {MSE_TARGET_OOF}). Worth submitting as a WAPE entry; won't take Overall.")
elif mse_competitive:
    print("~  TiDE has Chronos-competitive MSE but WAPE regressed.")
    print("   Check whether a TiDE + gated-MLP blend lifts both axes.")
elif blend_useful:
    print(f"✗  TiDE solo doesn't beat gated-MLP, MSE doesn't beat target.")
    print(f"   BUT residuals decorrelated (r={r_pooled:.3f} < 0.85). Useful 3rd arm for a blend.")
else:
    print(f"✗  TiDE solo {tide_arr.mean():.3f}, MSE {TIDE_MSE:.3f}, r {r_pooled:.3f}.")
    print("   With proper training applied and still no improvement, the architecture's")
    print("   inductive bias genuinely doesn't fit this dataset. Phase 8 closes here.")